In [ ]:
import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from IPython.display import HTML, display
from portfolio.signals import build_signal_table

# =========================================================================
# 1. FETCH ALL SIGNALS
# =========================================================================
results = build_signal_table()

# =========================================================================
# 2. BUILD HTML TABLE
# =========================================================================

def buy_color(signal):
    if "BUY NOW" in signal: return "#1B5E20"
    if "BUY DIP" in signal: return "#E65100"
    if "DCA" in signal: return "#1565C0"
    return "#B71C1C"

def sell_color(action):
    if "SELL NOW" in action: return "#B71C1C"
    if "NEAR TARGET" in action: return "#E65100"
    if "HOLD FOREVER" in action: return "#1565C0"
    if "HOLD" in action: return "#1B5E20"
    return "#1a1a1a"

def trend_color(trend):
    if trend == "UPTREND": return "#1B5E20"
    if trend == "DOWNTREND": return "#B71C1C"
    return "#E65100"

def basket_bg(basket):
    colors = {
        "Core ETF": "#E3F2FD",
        "Nuclear": "#FFF8DC",
        "Quantum": "#F3E6F5",
        "Quantum (paused)": "#F3E6F5",
        "Cyber": "#FFEBEE",
        "Industrial": "#E8EAF6",
        "SpecGrowth": "#E0F7FA",
    }
    return colors.get(basket, "#FFFFFF")

def fmt_price(val):
    if val is None: return "—"
    if val >= 1000: return f"${val:,.0f}"
    return f"${val:.0f}"

html = []

html.append("""<style>
.sig-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 12px; }
.sig-table th { background: #2C3E50; color: white; padding: 8px 10px; text-align: left; font-weight: bold; white-space: nowrap; }
.sig-table td { padding: 6px 10px; border-bottom: 1px solid #ddd; color: #1a1a1a; white-space: nowrap; }
.sig-table tr:hover { filter: brightness(0.95); }
.sig-header { font-size: 18px; font-weight: bold; margin-bottom: 8px; color: #2C3E50; }
.sig-sub { font-size: 12px; color: #555; margin-bottom: 16px; }
.sig-legend { font-size: 11px; color: #555; margin-top: 12px; line-height: 1.8; }
</style>""")

html.append('<div class="sig-header">Portfolio Buy / Sell Signal Dashboard</div>')
html.append('<div class="sig-sub">Targets based on analyst consensus + technicals as of June 2026. Signals update live.</div>')

html.append('<table class="sig-table">')
html.append('<tr>')
html.append('<th>Ticker</th><th>Name</th><th>Basket</th><th>Shares</th>')
html.append('<th>Current Price</th><th>Buy Below (live)</th><th>Sell Above</th>')
html.append('<th>RSI</th><th>Trend</th>')
html.append('<th>Buy Signal</th><th>Buy Reason</th>')
html.append('<th>Sell Signal</th><th>Sell Detail</th>')
html.append('<th>Catalyst / Thesis</th>')
html.append('</tr>')

basket_order = ["Core ETF", "Nuclear", "Quantum", "Quantum (paused)", "Cyber", "Industrial", "SpecGrowth"]
sorted_results = sorted(results, key=lambda r: (
    basket_order.index(r["basket"]) if r["basket"] in basket_order else 99,
    r["ticker"]
))

for r in sorted_results:
    bg = basket_bg(r["basket"])
    bc = buy_color(r["buy_signal"])
    sc = sell_color(r["sell_action"])
    tc = trend_color(r["trend"])

    # Current price formatting
    price = r["price"]
    price_str = f"${price:,.2f}" if price >= 100 else f"${price:.2f}"

    # Buy/sell target formatting
    buy_str = fmt_price(r["buy_target"])
    sell_str = fmt_price(r["sell_target"])

    # Upside % to sell target
    if r["sell_target"] and r["sell_target"] > price:
        upside = ((r["sell_target"] - price) / price) * 100
        sell_str += f" (+{upside:.0f}%)"

    html.append(f'<tr style="background:{bg};">')
    yf_url = f'https://finance.yahoo.com/quote/{r["ticker"]}/'
    html.append(f'<td><b><a href="{yf_url}" target="_blank" style="color:#1565C0;">{r["ticker"]}</a></b></td>')
    html.append(f'<td>{r["name"]}</td>')
    html.append(f'<td>{r["basket"]}</td>')
    html.append(f'<td style="text-align:center;">{r["shares"]}</td>')
    html.append(f'<td><b>{price_str}</b></td>')
    html.append(f'<td>{buy_str}</td>')
    html.append(f'<td>{sell_str}</td>')
    html.append(f'<td style="text-align:center;">{r["rsi"]:.0f}</td>')
    html.append(f'<td style="color:{tc}; font-weight:bold;">{r["trend"]}</td>')
    html.append(f'<td style="color:{bc}; font-weight:bold;">{r["buy_signal"]}</td>')
    html.append(f'<td>{r["buy_reason"]}</td>')
    html.append(f'<td style="color:{sc}; font-weight:bold;">{r["sell_action"]}</td>')
    html.append(f'<td>{r["sell_detail"]}</td>')
    html.append(f'<td style="max-width:280px; white-space:normal;">{r["catalyst"]}</td>')
    html.append('</tr>')

html.append('</table>')

html.append('<div class="sig-legend">')
html.append('<b>Buy Signals:</b> ')
html.append('<span style="color:#1565C0; font-weight:bold;">DCA</span> = buy monthly at any price (ETFs) | ')
html.append('<span style="color:#1B5E20; font-weight:bold;">BUY NOW</span> = at/near buy target or oversold | ')
html.append('<span style="color:#E65100; font-weight:bold;">BUY DIP</span> = scale in on weakness | ')
html.append('<span style="color:#B71C1C; font-weight:bold;">WAIT</span> = overbought or too far above target')
html.append('<br><b>Sell Signals:</b> ')
html.append('<span style="color:#1565C0; font-weight:bold;">HOLD FOREVER</span> = core ETF, never sell | ')
html.append('<span style="color:#1B5E20; font-weight:bold;">HOLD</span> = below target, keep holding | ')
html.append('<b>SELL @ EVENT</b> = sell on catalyst date | ')
html.append('<span style="color:#E65100; font-weight:bold;">NEAR TARGET</span> = within 10%, tighten stop | ')
html.append('<span style="color:#B71C1C; font-weight:bold;">SELL NOW</span> = at/above target')
html.append('<br><b>Buy Below (live):</b> computed dynamically from 50/200-SMA support + 52w low — updates every run')
html.append('<br><b>Sell Above:</b> take-profit price — sell at or above this level (with % upside from current)')
html.append('<br>⚠️ This is NOT financial advice. Targets are estimates based on analyst consensus and fundamentals.')
html.append('</div>')

display(HTML('\n'.join(html)))
print('\nSignal dashboard rendered.')







